# From Domain Features to Automated Feature Selection

## Learning outcomes

By the end of this notebook, you should be able to:

- Create reproducible manual features.
- Integrate feature construction into a pipeline.
- Distinguish feature engineering from feature selection.
- Compare filter and wrapper selection methods.
- Evaluate predictive performance, stability, and computational cost.

## 1. Feature engineering versus feature selection

**Feature engineering** creates candidate inputs for a model. It turns domain knowledge into measurable variables: ratios, differences, calendar effects, interaction terms, nonlinear transformations, and aggregations.

**Feature selection** chooses a subset of candidate inputs. It tries to keep signal while removing redundancy, noise, computational cost, and sources of overfitting.

A useful feature usually has at least one of these properties:

- It carries **signal** about the target.
- It is not just a duplicate of another feature.
- It is stable across samples, folds, and time periods.
- It is available at the moment the model is used.
- It does not encode the answer through leakage.

Feature engineering introduces hypotheses about the problem; feature selection controls complexity. Neither should inspect validation or test data.

## 2. Flight delay prediction: when can the model be used?

In this notebook we use a synthetic flight delay dataset. The target is whether a flight has an arrival delay of at least 15 minutes.

A central question is: **when do we want to make the prediction?**

- If the model is used **when the ticket is sold**, we can use route, carrier, season, day of week, and scheduled departure time. We cannot use weather observed on the departure day or airport congestion observed immediately before departure.
- If the model is used **the morning of travel**, we can use weather forecasts and known schedule pressure, but not actual departure delay.
- If the model is used **at the gate before departure**, we can use current weather and inbound aircraft status if available at that time.
- If the model is used **after departure**, actual departure delay may be available, but this is a different operational use case.

This notebook assumes the prediction is made shortly before scheduled departure. We therefore use scheduled time, route, carrier, aircraft, previous airport congestion, and weather forecast features. The raw table also contains `actual_departure` and `actual_arrival` so that we can discuss them with students, but the default modeling pipeline drops them. If we transform them into actual departure delay, actual arrival delay, or actual elapsed time, we are using information that is only known after the flight has already operated.

## 3. Setup

The setup cell imports modeling, preprocessing, feature selection, and interpretation utilities used throughout the lab.


In [ ]:
# Import the tools needed for reproducible feature engineering, selection, and evaluation.
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import RFE, SelectKBest, VarianceThreshold, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score, make_scorer, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

## 4. Create a raw mixed-type dataset

The dataset is generated inside the notebook so that the experiment is reproducible and does not depend on an external download. It contains numerical, categorical, scheduled date/time attributes, and actual date/time attributes.

The target is intentionally noisy: feature selection is more realistic when there is signal, redundancy, and noise at the same time.

`actual_departure` and `actual_arrival` are included as a teaching device. For a pre-departure delay prediction model, they should be dropped because they are not available when the prediction is made.

Students should read this generator as a data story: some variables are available before departure, while others are future information.


In [ ]:
# Generate a synthetic flight dataset with both safe scheduled fields and leaky actual-time fields.
def make_flight_delay_data(n_samples=3500, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    start = pd.Timestamp("2025-01-01")

    carriers = np.array(["ALP", "BLU", "CTY", "DTA", "ECO"])
    origins = np.array(["BOS", "JFK", "LAX", "ORD", "SFO", "SEA"])
    destinations = np.array(["ATL", "DEN", "DFW", "MIA", "MSP", "PHX"])
    aircraft = np.array(["A320", "A321", "B737", "B738", "E190"])

    day_offset = rng.integers(0, 365, size=n_samples)
    sched_hour = rng.choice(np.arange(5, 23), size=n_samples, p=np.array([0.03, 0.04, 0.05, 0.06, 0.06, 0.07, 0.08, 0.08, 0.08, 0.07, 0.07, 0.06, 0.06, 0.06, 0.05, 0.04, 0.03, 0.01]))
    sched_minute = rng.choice([0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55], size=n_samples)
    scheduled_departure = start + pd.to_timedelta(day_offset, unit="D") + pd.to_timedelta(sched_hour, unit="h") + pd.to_timedelta(sched_minute, unit="m")

    X = pd.DataFrame({
        "scheduled_departure": scheduled_departure,
        "carrier": rng.choice(carriers, size=n_samples, p=[0.22, 0.26, 0.18, 0.20, 0.14]),
        "origin": rng.choice(origins, size=n_samples),
        "destination": rng.choice(destinations, size=n_samples),
        "aircraft_type": rng.choice(aircraft, size=n_samples),
        "scheduled_duration_min": rng.normal(150, 45, size=n_samples).clip(35, 360),
        "distance_km": rng.normal(1150, 480, size=n_samples).clip(150, 4200),
        "prev_hour_origin_departures": rng.poisson(18, size=n_samples),
        "weather_wind_kmh": rng.gamma(2.2, 8.0, size=n_samples),
        "weather_precip_mm": rng.gamma(0.8, 2.2, size=n_samples),
        "visibility_km": rng.normal(13, 4, size=n_samples).clip(0.5, 25),
        "baggage_load_kg": rng.normal(4200, 900, size=n_samples).clip(800, 8500),
        "noise_random_1": rng.normal(0, 1, size=n_samples),
        "noise_random_2": rng.normal(0, 1, size=n_samples),
    })

    weekend = X["scheduled_departure"].dt.dayofweek.isin([4, 5, 6]).astype(int)
    evening = X["scheduled_departure"].dt.hour.between(17, 21).astype(int)
    bad_weather = (0.04 * X["weather_wind_kmh"] + 0.18 * X["weather_precip_mm"] - 0.05 * X["visibility_km"])
    congestion = 0.065 * X["prev_hour_origin_departures"]
    carrier_effect = X["carrier"].map({"ALP": -0.25, "BLU": 0.10, "CTY": 0.25, "DTA": 0.00, "ECO": 0.35})
    origin_effect = X["origin"].map({"BOS": 0.05, "JFK": 0.35, "LAX": 0.10, "ORD": 0.40, "SFO": 0.15, "SEA": 0.05})
    long_flight_pressure = 0.004 * (X["scheduled_duration_min"] - 150)
    interaction = 0.035 * X["weather_precip_mm"] * evening

    logit = -3.15 + bad_weather + congestion + carrier_effect + origin_effect + 0.45 * weekend + 0.55 * evening + long_flight_pressure + interaction
    probability = 1 / (1 + np.exp(-logit))
    y = rng.binomial(1, probability)

    scheduled_arrival = X["scheduled_departure"] + pd.to_timedelta(X["scheduled_duration_min"], unit="m")
    actual_departure_delay_min = np.where(
        y == 1,
        rng.normal(28, 18, size=n_samples),
        rng.normal(3, 8, size=n_samples),
    ).clip(-15, 120)
    actual_arrival_delay_min = (
        actual_departure_delay_min
        + np.where(y == 1, rng.normal(10, 12, size=n_samples), rng.normal(-2, 7, size=n_samples))
        + 0.4 * X["weather_precip_mm"]
    ).clip(-25, 180)
    actual_arrival_delay_min = np.where(y == 1, np.maximum(actual_arrival_delay_min, 15), np.minimum(actual_arrival_delay_min, 14))
    X["actual_departure"] = X["scheduled_departure"] + pd.to_timedelta(actual_departure_delay_min, unit="m")
    X["actual_arrival"] = scheduled_arrival + pd.to_timedelta(actual_arrival_delay_min, unit="m")

    return X, pd.Series(y, name="arrival_delay_15min")


X, y = make_flight_delay_data()
display(X.head())
print(y.value_counts(normalize=True).rename("class_share"))

## 5. Manual feature engineering

The following transformer creates domain-inspired features from raw columns:

- `departure_hour`, `day_of_week`, `month`, `is_weekend`, and `is_evening_peak`: scheduled date/time decomposition.
- `km_per_minute`: a ratio between distance and scheduled duration.
- `congestion_per_gate_proxy`: a normalized congestion proxy.
- `precip_x_evening_peak`: an interaction between bad weather and an operationally busy period.
- `low_visibility`: a nonlinear threshold feature.

Each of these encodes a hypothesis. For example, evening flights may accumulate delays from earlier flights, and precipitation may be worse during peak operating hours.

The transformer also has an `include_actual_times` option. It is disabled by default because actual departure and arrival times are not known before departure. Turn it on only to demonstrate leakage.

This transformer is the main teaching object. It turns domain assumptions into reproducible columns that can live safely inside a pipeline.


In [ ]:
# Define a sklearn-compatible transformer so feature engineering is fitted inside pipelines.
class FlightFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, add_manual_features=True, include_actual_times=False, drop_datetime=True):
        self.add_manual_features = add_manual_features
        self.include_actual_times = include_actual_times
        self.drop_datetime = drop_datetime

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_out = X.copy()
        scheduled_departure = pd.to_datetime(X_out["scheduled_departure"])

        # Minimal datetime decomposition is needed because most sklearn models do not accept datetime columns directly.
        X_out["departure_hour"] = scheduled_departure.dt.hour
        X_out["day_of_week"] = scheduled_departure.dt.dayofweek
        X_out["month"] = scheduled_departure.dt.month

        if self.include_actual_times:
            actual_departure = pd.to_datetime(X_out["actual_departure"])
            actual_arrival = pd.to_datetime(X_out["actual_arrival"])
            scheduled_arrival = scheduled_departure + pd.to_timedelta(X_out["scheduled_duration_min"], unit="m")
            X_out["actual_departure_hour"] = actual_departure.dt.hour
            X_out["actual_arrival_hour"] = actual_arrival.dt.hour
            X_out["actual_departure_delay_min"] = (actual_departure - scheduled_departure).dt.total_seconds() / 60
            X_out["actual_arrival_delay_min"] = (actual_arrival - scheduled_arrival).dt.total_seconds() / 60
            X_out["actual_elapsed_min"] = (actual_arrival - actual_departure).dt.total_seconds() / 60

        if self.add_manual_features:
            X_out["is_weekend"] = X_out["day_of_week"].isin([4, 5, 6]).astype(int)
            X_out["is_evening_peak"] = X_out["departure_hour"].between(17, 21).astype(int)
            X_out["km_per_minute"] = X_out["distance_km"] / X_out["scheduled_duration_min"].clip(lower=1)
            X_out["congestion_per_gate_proxy"] = X_out["prev_hour_origin_departures"] / 12
            X_out["precip_x_evening_peak"] = X_out["weather_precip_mm"] * X_out["is_evening_peak"]
            X_out["low_visibility"] = (X_out["visibility_km"] < 8).astype(int)

        if self.drop_datetime:
            X_out = X_out.drop(columns=["scheduled_departure", "actual_departure", "actual_arrival"])

        return X_out

**Dataframe schema for feature discussion**

Before fitting models, inspect the raw and engineered schema. The goal is not only to know the data types, but to decide which columns are available at prediction time and which ones would leak future information.

Use the schema table to ask: can this column exist at prediction time, and did we create it manually or from future data?


In [ ]:
# Compare safe and leaky engineered schemas before fitting any model.
safe_engineered_view = FlightFeatureEngineer(add_manual_features=True, include_actual_times=False).transform(X.head(5))
leaky_engineered_view = FlightFeatureEngineer(add_manual_features=True, include_actual_times=True).transform(X.head(5))

manual_feature_names = {
    "departure_hour", "day_of_week", "month", "is_weekend", "is_evening_peak",
    "km_per_minute", "congestion_per_gate_proxy", "precip_x_evening_peak", "low_visibility",
}
actual_time_feature_names = {
    "actual_departure", "actual_arrival", "actual_departure_hour", "actual_arrival_hour",
    "actual_departure_delay_min", "actual_arrival_delay_min", "actual_elapsed_min",
}

availability_notes = {
    "scheduled_departure": "known before departure",
    "actual_departure": "known only after departure",
    "actual_arrival": "known only after arrival",
    "actual_departure_hour": "known only after departure",
    "actual_arrival_hour": "known only after arrival",
    "actual_departure_delay_min": "known only after departure",
    "actual_arrival_delay_min": "known only after arrival; directly related to target",
    "actual_elapsed_min": "known only after arrival",
}

all_schema_columns = list(dict.fromkeys(list(X.columns) + list(leaky_engineered_view.columns)))
schema_rows = []

for column in all_schema_columns:
    if column in X.columns:
        dtype = X[column].dtype
        source = "raw"
    else:
        dtype = leaky_engineered_view[column].dtype
        source = "engineered"

    if column in actual_time_feature_names:
        keep_for_predeparture = "drop"
        leakage_risk = "high"
    elif column in safe_engineered_view.columns or column in X.columns:
        keep_for_predeparture = "candidate"
        leakage_risk = "low if measured before prediction"
    else:
        keep_for_predeparture = "review"
        leakage_risk = "unknown"

    schema_rows.append({
        "feature": column,
        "dtype": str(dtype),
        "source": source,
        "feature_family": "actual-time leakage" if column in actual_time_feature_names else "manual/domain" if column in manual_feature_names else "raw input",
        "availability": availability_notes.get(column, "known before scheduled departure in this example"),
        "predeparture_decision": keep_for_predeparture,
        "leakage_risk": leakage_risk,
    })

feature_schema = pd.DataFrame(schema_rows)
display(feature_schema)

print("Safe engineered dataframe shape:", safe_engineered_view.shape)
print("With actual-time features shape:", leaky_engineered_view.shape)

## 6. Baseline preprocessing pipeline

`ColumnTransformer` lets us apply different preprocessing to numerical and categorical columns after feature engineering. The selectors below infer column groups from the transformed dataframe, so the same pipeline works with and without the manual features.

The preprocessor discovers numerical and categorical columns after feature engineering, keeping the pipeline flexible.


In [ ]:
# Build reusable preprocessing and classifier helpers for the later experiments.
def numeric_columns(X):
    return X.select_dtypes(include=np.number).columns.tolist()


def categorical_columns(X):
    return X.select_dtypes(exclude=np.number).columns.tolist()


def make_preprocessor():
    try:
        one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        one_hot = OneHotEncoder(handle_unknown="ignore", sparse=False)

    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([("variance", VarianceThreshold()), ("scaler", StandardScaler())]), numeric_columns),
            ("cat", one_hot, categorical_columns),
        ],
        verbose_feature_names_out=False,
    )


def make_logistic(max_iter=1500):
    return LogisticRegression(max_iter=max_iter, solver="liblinear", class_weight="balanced")


def mutual_info_reproducible(X_array, y_array):
    return mutual_info_classif(X_array, y_array, random_state=RANDOM_STATE)


def make_pipeline(add_manual_features=False, include_actual_times=False, selector=None):
    steps = [
        ("feature_engineering", FlightFeatureEngineer(add_manual_features=add_manual_features, include_actual_times=include_actual_times)),
        ("preprocessing", make_preprocessor()),
    ]
    if selector is not None:
        steps.append(("feature_selection", selector))
    steps.append(("classifier", make_logistic()))
    return Pipeline(steps)

## 7. Automated feature selection

We compare two families of selectors:

- **Filter methods** rank features using a statistical criterion independent of the final model. Here we use mutual information with `SelectKBest`.
- **Wrapper methods** search for a subset by repeatedly fitting a model. Here we use `SequentialFeatureSelector` and `RFE`.

Filter methods are usually faster. Wrapper methods can capture model-specific usefulness, but they are more computationally expensive.

Important: selection is inside the pipeline. During cross-validation, the selector is fitted only on each training fold.

Selection is part of model fitting. Keeping it inside the pipeline prevents validation-fold information from guiding the selected features.


In [ ]:
# Configure one filter selector and two wrapper selectors for comparison.
from sklearn.feature_selection import SequentialFeatureSelector

filter_selector = SelectKBest(score_func=mutual_info_reproducible, k=20)

wrapper_selector = SequentialFeatureSelector(
    estimator=make_logistic(max_iter=800),
    n_features_to_select=20,
    direction="forward",
    scoring="roc_auc",
    cv=3,
    n_jobs=-1,
)

rfe_selector = RFE(
    estimator=make_logistic(max_iter=800),
    n_features_to_select=20,
    step=0.2,
)

## 8. Compare raw, engineered, and selected feature sets

We measure predictive performance, number of selected features, and training time. We also keep the fitted estimators from cross-validation so we can inspect selection stability.

The experiment table compares predictive value, selected feature count, and runtime rather than optimizing only one number.


In [ ]:
# Run every feature strategy under the same stratified cross-validation protocol.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "roc_auc": "roc_auc",
    "balanced_accuracy": make_scorer(balanced_accuracy_score),
    "f1": make_scorer(f1_score),
}

experiments = {
    "raw features": make_pipeline(add_manual_features=False),
    "manual features": make_pipeline(add_manual_features=True),
    "filter selection": make_pipeline(add_manual_features=False, selector=filter_selector),
    "wrapper selection": make_pipeline(add_manual_features=False, selector=wrapper_selector),
    "manual + filter selection": make_pipeline(add_manual_features=True, selector=SelectKBest(score_func=mutual_info_reproducible, k=20)),
    "manual + RFE": make_pipeline(add_manual_features=True, selector=rfe_selector),
}


def count_output_features(estimator):
    preprocessor = estimator.named_steps["preprocessing"]
    n_preprocessed = len(preprocessor.get_feature_names_out())
    selector = estimator.named_steps.get("feature_selection")
    if selector is None:
        return n_preprocessed
    return int(selector.get_support().sum())


rows = []
cv_results_by_name = {}

for name, estimator in experiments.items():
    start_time = time.perf_counter()
    result = cross_validate(estimator, X, y, cv=cv, scoring=scoring, return_estimator=True, n_jobs=-1)
    elapsed = time.perf_counter() - start_time
    cv_results_by_name[name] = result

    selected_counts = [count_output_features(fitted) for fitted in result["estimator"]]
    rows.append({
        "experiment": name,
        "roc_auc_mean": result["test_roc_auc"].mean(),
        "roc_auc_std": result["test_roc_auc"].std(),
        "balanced_accuracy_mean": result["test_balanced_accuracy"].mean(),
        "f1_mean": result["test_f1"].mean(),
        "features_mean": np.mean(selected_counts),
        "features_std": np.std(selected_counts),
        "fit_time_total_sec": elapsed,
    })

summary = pd.DataFrame(rows).sort_values("roc_auc_mean", ascending=False)
display(summary)

## 9. Actual departure and arrival times: keep or drop?

For this pre-departure prediction task, `actual_departure` and `actual_arrival` should be dropped. They are measured after the moment when the model is supposed to make its prediction.

However, they are useful teaching columns because they let us ask a precise operational question: **what information exists at prediction time?**

- Keep scheduled departure time: it is known before the flight.
- Usually drop actual departure time: it is known only after pushback/takeoff.
- Drop actual arrival time: it is known only after the flight has arrived.
- Drop engineered features derived from actual times, such as actual departure delay, actual arrival delay, or actual elapsed time.

The next cell intentionally compares a leakage-safe pipeline with a pipeline that uses actual-time-derived features. If the second pipeline looks much better, that is a warning sign, not a success.

The leakage comparison should look suspiciously strong when actual-time information is included.


In [ ]:
# Demonstrate how future information can inflate validation metrics.
leakage_demo = {
    "safe pre-departure features": make_pipeline(add_manual_features=True),
    "DO NOT USE: actual-time features": make_pipeline(add_manual_features=True, include_actual_times=True),
}

leakage_rows = []
for name, estimator in leakage_demo.items():
    result = cross_validate(estimator, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    leakage_rows.append({
        "experiment": name,
        "roc_auc_mean": result["test_roc_auc"].mean(),
        "balanced_accuracy_mean": result["test_balanced_accuracy"].mean(),
        "f1_mean": result["test_f1"].mean(),
    })

display(pd.DataFrame(leakage_rows))

## 10. Selection stability across folds

A selector is more trustworthy when it keeps similar features across folds. Instability does not always mean the selector is wrong: correlated features may be interchangeable. But instability should make us cautious when interpreting selected variables as domain truth.

Stability is an interpretation check: selected features that change across folds should be discussed cautiously.


In [ ]:
# Measure how similar the selected feature sets are across validation folds.
def selected_feature_names(fitted_pipeline):
    names = fitted_pipeline.named_steps["preprocessing"].get_feature_names_out()
    selector = fitted_pipeline.named_steps.get("feature_selection")
    if selector is None:
        return set(names)
    return set(names[selector.get_support()])


def mean_jaccard(selected_sets):
    scores = []
    for i in range(len(selected_sets)):
        for j in range(i + 1, len(selected_sets)):
            union = selected_sets[i] | selected_sets[j]
            intersection = selected_sets[i] & selected_sets[j]
            scores.append(len(intersection) / len(union))
    return np.mean(scores)


stability_rows = []
for name, result in cv_results_by_name.items():
    if "selection" not in name and "RFE" not in name:
        continue
    selected_sets = [selected_feature_names(fitted) for fitted in result["estimator"]]
    stability_rows.append({
        "experiment": name,
        "mean_jaccard_similarity": mean_jaccard(selected_sets),
        "example_selected_features": sorted(list(selected_sets[0]))[:15],
    })

stability = pd.DataFrame(stability_rows).sort_values("mean_jaccard_similarity", ascending=False)
display(stability)

## 11. Performance versus number of features

A useful visualization is a performance-versus-number-of-features curve. Here we vary `k` for `SelectKBest`. A good choice is not necessarily the highest possible score: we may prefer a smaller model when the performance loss is negligible.

This curve helps students look for diminishing returns as more features are selected.


In [ ]:
# Vary the number of selected features to look for diminishing returns.
k_values = [5, 10, 15, 20, 30, 40]
curve_rows = []

for k in k_values:
    estimator = make_pipeline(
        add_manual_features=True,
        selector=SelectKBest(score_func=mutual_info_reproducible, k=k),
    )
    result = cross_validate(estimator, X, y, cv=cv, scoring={"roc_auc": "roc_auc"}, n_jobs=-1)
    curve_rows.append({
        "k": k,
        "roc_auc_mean": result["test_roc_auc"].mean(),
        "roc_auc_std": result["test_roc_auc"].std(),
    })

curve = pd.DataFrame(curve_rows)
display(curve)

plt.figure(figsize=(7, 4))
plt.errorbar(curve["k"], curve["roc_auc_mean"], yerr=curve["roc_auc_std"], marker="o", capsize=4)
plt.xlabel("Number of selected features")
plt.ylabel("Cross-validated ROC AUC")
plt.title("Performance versus number of selected features")
plt.grid(alpha=0.3)
plt.show()

## 12. Interpretability check

After choosing a pipeline, fit it once on a training split and inspect which features are selected. This final inspection is for interpretation, not for deciding the reported cross-validation score.

Fit once on a holdout split only after cross-validation has motivated a candidate pipeline.


In [ ]:
# Fit the chosen pipeline once for holdout evaluation and selected-feature inspection.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

chosen_pipeline = make_pipeline(
    add_manual_features=True,
    selector=SelectKBest(score_func=mutual_info_reproducible, k=20),
)
chosen_pipeline.fit(X_train, y_train)

test_proba = chosen_pipeline.predict_proba(X_test)[:, 1]
test_pred = chosen_pipeline.predict(X_test)
print("Holdout ROC AUC:", round(roc_auc_score(y_test, test_proba), 3))
print("Holdout balanced accuracy:", round(balanced_accuracy_score(y_test, test_pred), 3))

selected = sorted(selected_feature_names(chosen_pipeline))
display(pd.DataFrame({"selected_feature": selected}))

Optional: permutation importance estimates how much the fitted pipeline depends on each raw input column. This is different from selected one-hot or engineered features, but it is useful for discussing model behavior at the original data level.

Permutation importance answers a different question from feature selection: which raw inputs the fitted pipeline relies on most.


In [ ]:
# Estimate how much each raw input column matters to the fitted pipeline.
importance = permutation_importance(
    chosen_pipeline,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance_table = (
    pd.DataFrame({
        "raw_feature": X_test.columns,
        "importance_mean": importance.importances_mean,
        "importance_std": importance.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
)
display(importance_table.head(12))

## 13. Student challenge

Create at least three domain-inspired features. For each feature, answer:

- What hypothesis does the feature encode?
- Does it improve cross-validated performance?
- Is the improvement consistent across folds?
- Does automated selection keep or remove it?
- Could the feature introduce leakage?

Ideas to try:

- A morning, afternoon, evening, and night departure period.
- A holiday-season indicator.
- Route-level historical delay rates, computed only inside training folds.
- A severe-weather indicator combining wind, precipitation, and visibility.
- Carrier-by-origin interaction categories.

Be careful with historical aggregates. If you compute route delay rate using the whole dataset before cross-validation, you leak validation-fold labels into the training features. To use target-based aggregates safely, implement them as a transformer fitted inside the pipeline.

The challenge asks students to connect feature ideas to evidence, not just to add columns.


In [ ]:
# Use this table to record feature hypotheses and evidence during the challenge.
# Challenge workspace
# 1. Extend FlightFeatureEngineer with your own features.
# 2. Re-run the experiments table.
# 3. Explain whether each feature was useful, stable, selected, and leakage-safe.

student_notes = pd.DataFrame({
    "feature": ["", "", ""],
    "hypothesis": ["", "", ""],
    "cv_improvement": ["", "", ""],
    "kept_by_selection": ["", "", ""],
    "leakage_risk": ["", "", ""],
})
display(student_notes)

## 14. Takeaway

Feature engineering is a way to encode assumptions about how the world works. Feature selection is a way to control complexity once we have many candidate features.

Both steps are part of model training. To obtain an honest estimate of performance, fit engineering steps that learn from data, preprocessing, and feature selection inside cross-validation, not before it.